# Diffusion-CCSP Option A/B on Google Colab

This notebook is set up for the new flow experiments:

- Option A: message-passing flow training via `train_flow_message_passing.py`
- Option B: iterative restart evaluation via `eval_flow_iterative.py`

Recommended Colab runtime: `GPU` with a `T4`.

The notebook is designed so you edit the config cell once, then run the remaining cells top to bottom.

In [ ]:
REPO_URL = 'https://github.com/ShirakuGIT/diffusion-ccsp.git'
REPO_DIR = '/content/diffusion-ccsp'
JACINLE_DIR = '/content/Jacinle'

USE_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive/diffusion_ccsp_colab'

DOWNLOAD_DATA = True
DOWNLOAD_BASELINE_CHECKPOINTS = True

ACTION = 'train_option_a'
# One of:
#   'train_option_a'
#   'eval_option_b_baseline'
#   'eval_option_b_message_passing'

INPUT_MODE = 'qualitative'
HIDDEN_DIM = 256
N_ROUNDS = 3
BATCH_SIZE = 64
TRAIN_STEPS = 200000
SAVE_EVERY = 10000
NUM_WORKERS = 2
PRINT_EVERY = 100
EVAL_N_SAMPLES = 3
EVAL_N_STEPS = 20

N_RESTARTS = 3
RESTART_NOISE = 0.10
NOISE_DECAY = 0.5
COMPARE_PURE = True
CHECKPOINT_TAG = 'best'

print('Configured ACTION =', ACTION)

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print(f'\n[run] {cmd}')
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)


if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    print('Drive root:', DRIVE_ROOT)

if not Path(REPO_DIR).exists():
    run(f'git clone {shlex.quote(REPO_URL)} {shlex.quote(REPO_DIR)}')
else:
    print('Repo already exists:', REPO_DIR)

if not Path(JACINLE_DIR).exists():
    run(f'git clone --recursive https://github.com/vacancy/Jacinle {shlex.quote(JACINLE_DIR)}')
else:
    print('Jacinle already exists:', JACINLE_DIR)

run('python -V')
run('nvidia-smi || true')

minimal_packages = [
    'gdown',
    'einops',
    'seaborn',
    'opencv-python',
    'pyglet==1.5.28',
    'pybullet',
    'pybullet-planning',
    'wandb',
    'torch-geometric',
]
run('pip install -q ' + ' '.join(minimal_packages))

os.environ['PYTHONPATH'] = ':'.join([
    str(Path(REPO_DIR)),
    str(Path(REPO_DIR) / 'envs'),
    str(Path(REPO_DIR) / 'networks'),
    str(JACINLE_DIR),
    os.environ.get('PYTHONPATH', ''),
])
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

print('PYTHONPATH configured.')

In [ ]:
import os
from pathlib import Path

if DOWNLOAD_DATA or DOWNLOAD_BASELINE_CHECKPOINTS:
    flags = []
    if DOWNLOAD_DATA:
        flags.append('--data')
    if DOWNLOAD_BASELINE_CHECKPOINTS:
        flags.append('--checkpoints')
    run(f"python download_data_checkpoints.py {' '.join(flags)}", cwd=REPO_DIR)
else:
    print('Skipping dataset/checkpoint download.')

run('python - <<\'PY\'\nfrom pathlib import Path\nroot = Path("data")\nfor p in sorted(root.glob("*")):\n    if p.is_dir():\n        print(p.name)\nPY', cwd=REPO_DIR)


## Run Option A or Option B

- `train_option_a`: trains the new message-passing flow model.
- `eval_option_b_baseline`: evaluates iterative restart flow on the baseline flow checkpoint.
- `eval_option_b_message_passing`: evaluates iterative restart flow on the trained message-passing checkpoint.

If `USE_DRIVE=True`, logs will also be written to your Drive path.

In [ ]:
from pathlib import Path

results_dir = None
if USE_DRIVE:
    results_dir = str(Path(DRIVE_ROOT) / f'flow_mp_{INPUT_MODE}_h{HIDDEN_DIM}_r{N_ROUNDS}')
    Path(results_dir).mkdir(parents=True, exist_ok=True)

if ACTION == 'train_option_a':
    cmd = [
        'python', 'train_flow_message_passing.py',
        '-input_mode', INPUT_MODE,
        '-hidden_dim', str(HIDDEN_DIM),
        '-n_rounds', str(N_ROUNDS),
        '-batch_size', str(BATCH_SIZE),
        '-train_num_steps', str(TRAIN_STEPS),
        '-save_every', str(SAVE_EVERY),
        '-device', 'cuda',
        '-num_workers', str(NUM_WORKERS),
        '-print_every', str(PRINT_EVERY),
        '-eval_n_samples', str(EVAL_N_SAMPLES),
        '-eval_n_steps', str(EVAL_N_STEPS),
        '--verbose',
    ]
    if results_dir is not None:
        cmd += ['-results_dir', results_dir]
    run(' '.join(shlex.quote(x) for x in cmd), cwd=REPO_DIR)

elif ACTION == 'eval_option_b_baseline':
    cmd = [
        'python', 'eval_flow_iterative.py',
        '--model', 'baseline',
        '--input_mode', INPUT_MODE,
        '--hidden_dim', str(HIDDEN_DIM),
        '--checkpoint_tag', CHECKPOINT_TAG,
        '--n_steps', str(EVAL_N_STEPS),
        '--n_restarts', str(N_RESTARTS),
        '--restart_noise', str(RESTART_NOISE),
        '--noise_decay', str(NOISE_DECAY),
        '--n_samples', str(EVAL_N_SAMPLES),
        '--device', 'cuda',
        '--verbose',
    ]
    if COMPARE_PURE:
        cmd.append('--compare_pure')
    run(' '.join(shlex.quote(x) for x in cmd), cwd=REPO_DIR)

elif ACTION == 'eval_option_b_message_passing':
    cmd = [
        'python', 'eval_flow_iterative.py',
        '--model', 'message_passing',
        '--input_mode', INPUT_MODE,
        '--hidden_dim', str(HIDDEN_DIM),
        '--n_rounds', str(N_ROUNDS),
        '--checkpoint_tag', CHECKPOINT_TAG,
        '--n_steps', str(EVAL_N_STEPS),
        '--n_restarts', str(N_RESTARTS),
        '--restart_noise', str(RESTART_NOISE),
        '--noise_decay', str(NOISE_DECAY),
        '--n_samples', str(EVAL_N_SAMPLES),
        '--device', 'cuda',
        '--verbose',
    ]
    if COMPARE_PURE:
        cmd.append('--compare_pure')
    if results_dir is not None:
        ckpt = str(Path(results_dir) / f'flow_model_{CHECKPOINT_TAG}.pt')
        cmd += ['--checkpoint', ckpt]
    run(' '.join(shlex.quote(x) for x in cmd), cwd=REPO_DIR)

else:
    raise ValueError(f'Unknown ACTION: {ACTION}')

In [ ]:
from pathlib import Path

print('\nUseful output locations:')
print('Repo logs:', Path(REPO_DIR) / 'logs')
if USE_DRIVE:
    print('Drive logs:', Path(DRIVE_ROOT))

if USE_DRIVE:
    run(f'find {shlex.quote(DRIVE_ROOT)} -maxdepth 3 | sed -n "1,200p"')